In [ ]:
%run Simulated_Annealing.ipynb
import random
import time
from collections import deque
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- CSS Styling ---
custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght=400;500;600;700&family=JetBrains+Mono&display=swap');

.app-container { background-color: #f9f9ff; font-family: 'Inter', sans-serif; }
.modern-card { background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); box-sizing: border-box; }
.card-header { font-size: 18px; font-weight: 600; color: #141b2b; border-bottom: 1px solid #c3c6d7; padding-bottom: 10px; margin-bottom: 15px; display: flex; justify-content: space-between; }

.stat-box { background-color: #ffffff !important; border: 1px solid #c3c6d7 !important; border-radius: 12px !important; padding: 15px !important; box-sizing: border-box; }

.puzzle-input input[type="number"] {
    font-size: 24px !important; font-weight: 700 !important; text-align: center !important; color: #004ac6 !important;
    background-color: #e1e8fd !important; border: 1px solid rgba(0,74,198,0.2) !important; border-radius: 8px !important;
    height: 100% !important; box-sizing: border-box;
}

.btn-primary { 
    background-color: #004ac6 !important; color: white !important; border-radius: 999px !important; 
    font-weight: 600 !important; border: 1px solid #004ac6 !important; width: 95% !important; box-sizing: border-box !important;
}
.btn-primary:hover { background-color: #003ea8 !important; }

.btn-action { background-color: #e1e8fd !important; color: #38485d !important; border-radius: 8px !important; font-weight: 600 !important; border: 1px solid rgba(0,74,198,0.2) !important; font-size: 12px !important; }

.log-output { background-color: #f1f3ff !important; font-family: 'JetBrains Mono', monospace !important; border: none !important; }

.anim-board { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; max-width: 320px; margin: 0 auto; background-color: #f1f3ff; padding: 20px; border-radius: 16px; }
.anim-tile { aspect-ratio: 1; background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 12px; display: flex; align-items: center; justify-content: center; font-size: 32px; font-weight: 700; color: #004ac6; box-shadow: 0 2px 5px rgba(0,0,0,0.05); transition: all 0.3s ease; }
.anim-tile-empty { aspect-ratio: 1; background-color: rgba(220, 226, 247, 0.4); border: 2px dashed #c3c6d7; border-radius: 12px; }
</style>
"""

display(HTML(custom_css))

header_html = widgets.HTML(value="""
<div style="display: flex; justify-content: space-between; align-items: center; padding: 15px 30px; background-color: #ffffff; border-bottom: 1px solid #c3c6d7; font-family: 'Inter', sans-serif;">
    <span style="font-size: 20px; font-weight: 700; color: #141b2b;">Simulated Annealing</span>
    <div style="color: #004ac6; font-weight: 700; border-bottom: 2px solid #004ac6; padding-bottom: 4px; font-size: 14px;">Simulated Annealing</div>
</div>
""")

input_boxes = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='60px')) 
               for v in [2, 8, 3, 1, 6, 4, 7, 0, 5]]
for box in input_boxes: box.add_class('puzzle-input')

input_grid = widgets.GridBox(input_boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="10px", margin="0 0 20px 0"))

btn_random = widgets.Button(description="Random", layout=widgets.Layout(flex='1'))
btn_random.add_class('btn-action')
btn_random.style.button_color = '#505f76'
btn_random.style.text_color = 'white'

btn_reset = widgets.Button(description="Reset", layout=widgets.Layout(flex='1'))
btn_reset.add_class('btn-action')
btn_load = widgets.Button(description="Load Example", layout=widgets.Layout(flex='1'))
btn_load.add_class('btn-action')

action_btns = widgets.HBox([btn_random, btn_reset, btn_load], layout=widgets.Layout(gap='10px'))

initial_state_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>Initial State</span></div>'),
    input_grid, action_btns
], layout=widgets.Layout(margin='0 0 20px 0'))
initial_state_card.add_class('modern-card')

btn_hc_search = widgets.Button(description="Simulated Annealing", layout=widgets.Layout(height='45px'))
btn_hc_search.add_class('btn-primary')

temp_input = widgets.BoundedFloatText(
    value=1000.0, min=1.0, max=100000.0, step=10.0,
    description='Nhiệt độ (T):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='0 0 10px 0')
)

cooling_input = widgets.BoundedFloatText(
    value=0.95, min=0.01, max=0.99, step=0.01,
    description='Hệ số giảm (α):', style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='0 0 15px 0')
)

config_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>Simulated Annealing</span></div>'),
    widgets.HTML('<div style="font-size:12px; color:#54647a; margin-bottom:12px; font-style:italic; text-align:center;">Heuristic: Manhattan Distance<br>Cho phép di chuyển xấu theo xác suất P = e^(-ΔE/T).<br>Nhiệt độ T giảm dần theo hệ số α mỗi bước.</div>'),
    temp_input, cooling_input,
    widgets.VBox([btn_hc_search], layout=widgets.Layout(width='100%', align_items='center'))
], layout=widgets.Layout(margin='0 0 20px 0')) 
config_card.add_class('modern-card')

step_label = widgets.HTML('<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Ready</span>')
sim_header = widgets.HBox([
    widgets.HTML('<span style="font-size: 18px; font-weight: 600; color: #141b2b;">Visual Simulation</span>'),
    step_label
], layout=widgets.Layout(justify_content='space-between', border_bottom='1px solid #c3c6d7', padding='0 0 15px 0', margin='0 0 20px 0', width='100%'))

anim_html = widgets.HTML(value="")
sim_card = widgets.VBox([sim_header, anim_html], layout=widgets.Layout(flex='1'))
sim_card.add_class('modern-card')

left_col = widgets.VBox([initial_state_card, sim_card], layout=widgets.Layout(flex='1.6', min_width='50%'))

stat_steps = widgets.HTML()
stat_nodes = widgets.HTML()
stat_time = widgets.HTML()
stat_status = widgets.HTML()
stat_temp = widgets.HTML()

def update_stat(html_widget, value):
    html_widget.value = f'<div style="font-size: 20px; font-weight: 700; color: #141b2b; text-align: center; height: 35px; display: flex; align-items: center; justify-content: center;">{value}</div>'

update_stat(stat_steps, "-")
update_stat(stat_nodes, "-")
update_stat(stat_time, "-")
update_stat(stat_status, "-")
update_stat(stat_temp, "-")

def make_stat_box(title, html_widget):
    box = widgets.VBox([
        widgets.HTML(f'<span style="font-size: 11px; font-weight: 600; color: #54647a; text-transform: uppercase; display: block; text-align: center; width: 100%;">{title}</span>'),
        html_widget
    ], layout=widgets.Layout(width='100%', align_items='center'))
    box.add_class('stat-box')
    return box

stat_grid = widgets.GridBox([
    make_stat_box("Steps", stat_steps),
    make_stat_box("Nodes", stat_nodes),
    make_stat_box("Time", stat_time),
    make_stat_box("T & α", stat_temp),
    make_stat_box("Status", stat_status)
], layout=widgets.Layout(grid_template_columns="1fr 1fr", gap="15px", margin="0 0 20px 0"))

log_content = widgets.HTML(value='')
out_text = widgets.VBox([log_content], layout=widgets.Layout(flex='1', overflow='auto', padding='15px', max_height='380px'))
out_text.add_class('log-output')

log_card = widgets.VBox([
    widgets.HTML('<div style="display:flex; align-items:center; justify-content:space-between; border-bottom: 1px solid #c3c6d7; padding: 12px 20px; background-color: #e1e8fd; border-radius: 12px 12px 0 0;"><span style="font-size: 13px; font-weight: 700; color: #141b2b; text-transform: uppercase; letter-spacing: 0.5px;">Execution Log</span></div>'),
    out_text
], layout=widgets.Layout(background_color='#f1f3ff', border='1px solid #c3c6d7', border_radius='12px', flex='1'))

right_col = widgets.VBox([stat_grid, config_card, log_card], layout=widgets.Layout(flex='1', min_width='330px'))

main_app = widgets.VBox([
    header_html,
    widgets.HBox([left_col, right_col], layout=widgets.Layout(padding='20px', gap='20px'))
])
main_app.add_class('app-container')

def print_log_state(title, state, action=None, h_cost=0, h_label="h"):
    state_str = ""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        state_str += "  " + "    ".join([str(x) if x != 0 else "[ ]" for x in row]) + "\n"
    if action:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">Bước {title}: {action} ({h_label} = <span style="color: #004ac6;">{h_cost}</span>)</p>'
    else:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">{title} ({h_label} = {h_cost})</p>'
    log_html = '<div style="margin-bottom: 15px;">' + action_html
    log_html += '<div style="background-color: #ffffff; padding: 12px; border-radius: 8px; border: 1px solid rgba(195, 198, 215, 0.5); display: inline-block;">'
    log_html += '<pre style="margin: 0; font-family: monospace; font-size: 13px; line-height: 1.4; color: #141b2b;">' + state_str + '</pre>'
    log_html += '</div></div><div style="border-top: 1px solid rgba(195, 198, 215, 0.5); margin-bottom: 15px; width: 100%;"></div>'
    log_content.value += log_html

def render_board(state):
    html_content = '<div class="anim-board">'
    for val in state:
        if val == 0:
            html_content += '<div class="anim-tile-empty"></div>'
        else:
            html_content += f'<div class="anim-tile">{val}</div>'
    html_content += '</div>'
    anim_html.value = html_content

def animate_path(start_state, path, goal_state):
    render_board(start_state)
    step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step 0/{len(path)}</span>'
    time.sleep(1)
    for step, (action, state) in enumerate(path):
        render_board(state)
        step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step {step + 1}/{len(path)}</span>'
        time.sleep(0.6)
    final_state = path[-1][1] if path else start_state
    if final_state == goal_state:
        step_label.value = '<span style="background:#d1f4e0; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#0d6e35;">Finished ✅</span>'
    else:
        step_label.value = '<span style="background:#fde8e8; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#c53030;">Failed ❌</span>'

def solve_and_animate():
    log_content.value = ''
    start_state = [box.value for box in input_boxes]
    goal_state = [1, 2, 3, 8, 0, 4, 7, 6, 5]
    initial_temp = temp_input.value
    cooling_rate = cooling_input.value
    render_board(start_state)
    log_content.value += f'<div style="color: #004ac6; font-weight: bold; margin-bottom: 15px; font-family: Inter; font-size: 14px;">Simulated Annealing (T={initial_temp}, α={cooling_rate})</div>'
    start_time = time.time()
    try:
        path, nodes_generated, log_data = simulated_annealing_solve(start_state, goal_state, initial_temp=initial_temp, cooling_rate=cooling_rate)
    except Exception as e:
        log_content.value += f'<div style="color: red;">❌ Lỗi: {str(e)}</div>'
        return
    end_time = time.time()
    elapsed_ms = int((end_time - start_time) * 1000)
    final_state = path[-1][1] if path else start_state
    success = (final_state == goal_state)
    status_text = '<span style="color: #0d6e35; font-weight: bold;">SUCCESS</span>' if success else '<span style="color: #c53030; font-weight: bold;">FAILED</span>'
    update_stat(stat_steps, str(len(path)))
    update_stat(stat_nodes, f"{nodes_generated:,}")
    update_stat(stat_time, f"{elapsed_ms}ms")
    update_stat(stat_temp, f"{initial_temp} | {cooling_rate}")
    update_stat(stat_status, status_text)
    for log_entry in log_data:
        step_val = log_entry['step']
        if isinstance(step_val, str) and step_val == 'KQ':
            bg_color = '#fde8e8'
            border_color = 'rgba(197, 48, 48, 0.3)'
            step_color = '#c53030'
        else:
            bg_color = '#fcfcfc'
            border_color = 'rgba(0,74,198,0.1)'
            step_color = '#004ac6'
        log_content.value += f'<div style="background-color: {bg_color}; padding: 12px; border-radius: 8px; border: 1px solid {border_color}; margin-bottom: 12px; font-family: Inter; font-size: 13px; line-height: 1.5;">'
        log_content.value += f'<div style="font-weight: 700; color: {step_color}; border-bottom: 1px dashed {border_color}; padding-bottom: 5px; margin-bottom: 8px;">BƯỚC {step_val}</div>'
        log_content.value += f'<div>{log_entry["action_html"]}</div>'
        log_content.value += f'<div style="margin-top: 8px; font-size: 12px; color: #54647a; background-color: #f1f3ff; padding: 4px 8px; border-radius: 4px;"><b>{log_entry["frontier_str"]}</b> | <b>{log_entry["reached_str"]}</b></div>'
        log_content.value += '</div>'
    if path:
        log_content.value += '<div style="margin: 20px 0; border-top: 2px solid #004ac6; padding-top: 15px;"><b style="color: #004ac6;">MA TRẬN CÁC BƯỚC ĐI:</b></div>'
        h_label = "h(Manhattan)"
        h_start = count_manhattan(start_state, goal_state)
        print_log_state("Bắt đầu", start_state, h_cost=h_start, h_label=h_label)
        for step_idx, (action, state) in enumerate(path):
            h_n = count_manhattan(state, goal_state)
            print_log_state(str(step_idx + 1), state, action, h_cost=h_n, h_label=h_label)
    if path:
        thread = threading.Thread(target=animate_path, args=(start_state, path, goal_state))
        thread.start()
    else:
        step_label.value = '<span style="background:#fde8e8; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#c53030;">No Solution Found</span>'

btn_hc_search.on_click(lambda b: solve_and_animate())

def randomize_board(b):
    nums = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums)
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
btn_random.on_click(randomize_board)

def reset_board(b):
    nums = [0]*9
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
btn_reset.on_click(reset_board)

def load_example(b):
    nums = [2, 8, 3, 1, 6, 4, 7, 0, 5]
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
btn_load.on_click(load_example)

render_board([2, 8, 3, 1, 6, 4, 7, 0, 5])
display(main_app)
